In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature = 0.0)

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from pytube import YouTube

def get_transcript(video_url):
    yt = YouTube(video_url)
    video_id = yt.video_id
    transcript = YouTubeTranscriptApi.get_transcript(video_id)
    full_text = " ".join([entry['text'] for entry in transcript])
    return full_text

In [12]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

video_url = "https://youtu.be/ylhcZZ7O3Tk?si=5lHtR0UsxaOMlD5A"
transcript_text = get_transcript(video_url)
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
docs = splitter.create_documents([transcript_text])

In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import Chroma
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
db = Chroma.from_documents(docs, embeddings)

In [14]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

qa = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = db.as_retriever()
)

In [17]:
import time
from langchain.prompts import PromptTemplate

chat_history = []
print(qa.input_keys)
def chat():
    global chat_history
    print("""
    What do you want to do:
    1. Ask a question
    2. Let me ask you 3 questions
    3. Exit the chat
    """)

    while True:
        a = input("Enter your choice: ")

        if a == "1":
            question = input("Enter your question: ")
            print("I'm thinking...")
            time.sleep(1)

            # ✅ Call QA system with question and history
            response = qa.invoke({
                "question": question,
                "chat_history": chat_history
            })

            # ✅ Append to history
            chat_history.append((question, response["answer"]))
            print("Answer:", response["answer"])

        elif a == "2":
            print("I'll ask you 3 questions to test your understanding...")

            # ✅ Generate test questions
            qa_prompt = PromptTemplate.from_template(
                "Hey, based on the video transcript: {transcript}, generate only 3 questions in english that can be asked to the user to test their understanding of the material."
            )
            prompt_text = qa_prompt.format_prompt(transcript=transcript_text).to_string()

            generated_questions = qa.invoke({
                "question": prompt_text,
                "chat_history": chat_history
            })["answer"]

            print("Answer the following questions:\n")
            print(generated_questions)

            user_response = input("\nYour answer: ")

            chat_history.append((generated_questions, user_response))

            # ✅ Generate model's answer
            model_answer = qa.invoke({
                "question": generated_questions,
                "chat_history": chat_history
            })["answer"]

            chat_history.append((generated_questions, model_answer))

            # ✅ Grading prompt
            grade_prompt = PromptTemplate.from_template(
                "You are a grading assistant.\n"
                "Question: {question}\n"
                "User Answer: {answer}\n"
                "Model Answer: {model_answer}\n"
                "Now, give a score out of 10 and a short reason."
            )

            grade_input = grade_prompt.format_prompt(
                question=generated_questions,
                answer=user_response,
                model_answer=model_answer
            ).to_string()

            grade_response = qa.invoke({
                "question": grade_input,
                "chat_history": chat_history
            })["answer"]

            chat_history.append((grade_input, grade_response))

            print("\nYour Score and Feedback:")
            print(grade_response)

        else:
            print("Goodbye! 👋")
            break

chat()


['question', 'chat_history']

    What do you want to do:
    1. Ask a question
    2. Let me ask you 3 questions
    3. Exit the chat
    
I'll ask you 3 questions to test your understanding...
Answer the following questions:

Okay, here are 3 questions to test understanding of the video transcript about variables:

1.  Explain in your own words what a variable is and why they are useful in programming, according to the video.
2.  The video mentions that variables can store different types of data. What two types of data are specifically mentioned, and how does the `+` operator behave differently with each of these data types?
3.  In the video, the value of the variable `time` is changed multiple times. Explain how the value of a variable can be changed and give an example from the video.

Your Score and Feedback:
8/10. The response provides a good explanation of variables, their assignment, and how their values can be changed. It also includes a relevant example. However, it could be

In [18]:
print(chat_history)

[('Okay, here are 3 questions to test understanding of the video transcript about variables:\n\n1.  Explain in your own words what a variable is and why they are useful in programming, according to the video.\n2.  The video mentions that variables can store different types of data. What two types of data are specifically mentioned, and how does the `+` operator behave differently with each of these data types?\n3.  In the video, the value of the variable `time` is changed multiple times. Explain how the value of a variable can be changed and give an example from the video.', '1.  According to the video, variables are used to keep track of things as they change throughout time in a program. They associate values with names that can be reused in the program.  2. The video mentions that with strings, you can concatenate them (combine them together), while with numbers, you can add or subtract them. The video does not provide a direct example of the `+` operator. 3.  The video explains tha